# Results
Aggregated analysis and figures for the paper.
Requires `results/fever_results.csv` and `results/hotpotqa_results.csv` (produced by notebooks 04 and 05).

In [ ]:
#to deletee

import time, torch, yaml, sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.generation.llm_client import HuggingFaceClient

with open("../configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

PROMPT = "Is the Earth round, explain in detail."
llm_cache = os.path.join("../", cfg["cache"]["dir"], cfg["cache"]["llm_subdir"])

for model in cfg["models"]["available"]:
    print(f"Loading {model}...", flush=True)
    llm = HuggingFaceClient(model=model, temperature=0.0, cache_dir=llm_cache)
    with llm:
        llm._load_model()
        input_text = llm._tokenizer.apply_chat_template(
            [{"role": "user", "content": PROMPT}],
            tokenize=False, add_generation_prompt=True,
        )
        inputs = llm._tokenizer(input_text, return_tensors="pt").to(llm._device)
        input_len = inputs["input_ids"].shape[1]
        t0 = time.perf_counter()
        with torch.no_grad():
            out = llm._hf_model.generate(**inputs, max_new_tokens=100, do_sample=False,
                                          pad_token_id=llm._tokenizer.eos_token_id)
        elapsed = time.perf_counter() - t0
        n = len(out[0]) - input_len
        print(f"  {n/elapsed:.1f} tok/s  ({n} tokens in {elapsed:.2f}s)")

Loading Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  22.9 tok/s  (100 tokens in 4.37s)
Loading google/gemma-2-2b-it...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

  14.3 tok/s  (100 tokens in 7.02s)
Loading HuggingFaceTB/SmolLM2-1.7B-Instruct...


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

  20.7 tok/s  (100 tokens in 4.82s)


In [8]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

from nb_style import MODEL_LABELS, MODEL_COLORS, PROMPT_COLORS, PROMPT_COLORS_QA

RESULTS_DIR = Path("../results")
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(exist_ok=True)

fever = pd.read_csv(RESULTS_DIR / "fever_results.csv")
hpqa  = pd.read_csv(RESULTS_DIR / "hotpotqa_results.csv")
print("FEVER rows:", len(fever))
print("HotpotQA rows:", len(hpqa))
fever.head()

FileNotFoundError: [Errno 2] No such file or directory: '../results/fever_results.csv'

## FEVER Results

In [ ]:
PROMPT_LABELS = {
    "standard":         "Standard",
    "chain_of_thought": "CoT",
    "vigilant":         "Vigilant",
}

fever["model_label"]  = fever["model"].map(MODEL_LABELS)
fever["prompt_label"] = fever["prompt_type"].map(PROMPT_LABELS)

pivot_acc = (
    fever
    .pivot_table(index=["model_label", "prompt_label"],
                 columns="poison_rate", values="accuracy")
    .rename(columns={0: "r=0 (clean)", 1: "r=1 (poisoned)"})
)
pivot_acc["delta"] = pivot_acc["r=1 (poisoned)"] - pivot_acc["r=0 (clean)"]
print("=== FEVER — Accuracy ===\n")
print(pivot_acc.round(3).to_string())

In [ ]:
pivot_f1 = (
    fever
    .pivot_table(index=["model_label", "prompt_label"],
                 columns="poison_rate", values="macro_f1")
    .rename(columns={0: "r=0 (clean)", 1: "r=1 (poisoned)"})
)
pivot_f1["delta"] = pivot_f1["r=1 (poisoned)"] - pivot_f1["r=0 (clean)"]
print("=== FEVER — Macro-F1 ===\n")
print(pivot_f1.round(3).to_string())

In [ ]:
# Bar chart: accuracy by model, grouped by prompt, for r=0 and r=1
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
models  = list(MODEL_LABELS.values())
prompts = list(PROMPT_LABELS.values())
x = np.arange(len(models))
width = 0.25

for ax, r, title in zip(axes, [0, 1], ["r=0 (clean)", "r=1 (poisoned)"]):
    sub = fever[fever["poison_rate"] == r]
    for i, (pt_key, pt_label) in enumerate(PROMPT_LABELS.items()):
        vals = [sub[(sub["model"] == m) & (sub["prompt_type"] == pt_key)]["accuracy"].values
                for m in MODEL_LABELS]
        heights = [v[0] if len(v) else float("nan") for v in vals]
        ax.bar(x + i * width, heights, width, label=pt_label,
               color=PROMPT_COLORS[pt_key], alpha=0.85, edgecolor="white")
    ax.set_title(title)
    ax.set_xticks(x + width)
    ax.set_xticklabels(models, rotation=15, ha="right")
    ax.set_ylabel("Accuracy")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.legend(title="Prompt")
    ax.set_ylim(0, 1)

fig.suptitle("FEVER — Accuracy by Model and Prompt Type", fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fever_accuracy.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Degradation: accuracy drop (r=1 - r=0) per model × prompt
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(models))
width = 0.25

for i, (pt_key, pt_label) in enumerate(PROMPT_LABELS.items()):
    deltas = []
    for m in MODEL_LABELS:
        r0 = fever[(fever["model"] == m) & (fever["prompt_type"] == pt_key)
                   & (fever["poison_rate"] == 0)]["accuracy"].values
        r1 = fever[(fever["model"] == m) & (fever["prompt_type"] == pt_key)
                   & (fever["poison_rate"] == 1)]["accuracy"].values
        deltas.append((r1[0] - r0[0]) if len(r0) and len(r1) else float("nan"))
    ax.bar(x + i * width, deltas, width, label=pt_label,
           color=PROMPT_COLORS[pt_key], alpha=0.85, edgecolor="white")

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(x + width)
ax.set_xticklabels(models, rotation=15, ha="right")
ax.set_ylabel("Accuracy delta (r=1 - r=0)")
ax.set_title("FEVER — Accuracy Degradation Under Poisoning")
ax.legend(title="Prompt")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fever_degradation.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Hallucination rate comparison
pivot_hal = (
    fever
    .pivot_table(index=["model_label", "prompt_label"],
                 columns="poison_rate", values="hallucination_rate")
    .rename(columns={0: "r=0", 1: "r=1"})
)
pivot_hal["delta"] = pivot_hal["r=1"] - pivot_hal["r=0"]
print("=== FEVER — Hallucination Rate ===\n")
print(pivot_hal.round(3).to_string())

## HotpotQA Results

In [ ]:
PROMPT_LABELS_QA = {
    "standard_qa": "Standard",
    "cot_qa":      "CoT",
    "vigilant_qa": "Vigilant",
}

hpqa["model_label"]  = hpqa["model"].map(MODEL_LABELS)
hpqa["prompt_label"] = hpqa["prompt_type"].map(PROMPT_LABELS_QA)

pivot_em = (
    hpqa
    .pivot_table(index=["model_label", "prompt_label"],
                 columns="poison_rate", values="exact_match")
    .rename(columns={0: "r=0 (clean)", 1: "r=1 (poisoned)"})
)
pivot_em["delta"] = pivot_em["r=1 (poisoned)"] - pivot_em["r=0 (clean)"]
print("=== HotpotQA — Exact Match ===\n")
print(pivot_em.round(3).to_string())

pivot_tf1 = (
    hpqa
    .pivot_table(index=["model_label", "prompt_label"],
                 columns="poison_rate", values="token_f1")
    .rename(columns={0: "r=0 (clean)", 1: "r=1 (poisoned)"})
)
pivot_tf1["delta"] = pivot_tf1["r=1 (poisoned)"] - pivot_tf1["r=0 (clean)"]
print("\n=== HotpotQA — Token F1 ===\n")
print(pivot_tf1.round(3).to_string())

In [ ]:
# HotpotQA bar charts: EM by model × prompt for r=0 and r=1
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
x = np.arange(len(models))

for ax, r, title in zip(axes, [0, 1], ["r=0 (clean)", "r=1 (poisoned)"]):
    sub = hpqa[hpqa["poison_rate"] == r]
    for i, (pt_key, pt_label) in enumerate(PROMPT_LABELS_QA.items()):
        vals = [sub[(sub["model"] == m) & (sub["prompt_type"] == pt_key)]["exact_match"].values
                for m in MODEL_LABELS]
        heights = [v[0] if len(v) else float("nan") for v in vals]
        ax.bar(x + i * width, heights, width, label=pt_label,
               color=PROMPT_COLORS_QA[pt_key], alpha=0.85, edgecolor="white")
    ax.set_title(title)
    ax.set_xticks(x + width)
    ax.set_xticklabels(models, rotation=15, ha="right")
    ax.set_ylabel("Exact Match")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.legend(title="Prompt")
    ax.set_ylim(0, 1)

fig.suptitle("HotpotQA — Exact Match by Model and Prompt Type", fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "hotpotqa_em.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# HotpotQA degradation (EM drop)
fig, ax = plt.subplots(figsize=(8, 4))

for i, (pt_key, pt_label) in enumerate(PROMPT_LABELS_QA.items()):
    deltas = []
    for m in MODEL_LABELS:
        r0 = hpqa[(hpqa["model"] == m) & (hpqa["prompt_type"] == pt_key)
                  & (hpqa["poison_rate"] == 0)]["exact_match"].values
        r1 = hpqa[(hpqa["model"] == m) & (hpqa["prompt_type"] == pt_key)
                  & (hpqa["poison_rate"] == 1)]["exact_match"].values
        deltas.append((r1[0] - r0[0]) if len(r0) and len(r1) else float("nan"))
    ax.bar(x + i * width, deltas, width, label=pt_label,
           color=PROMPT_COLORS_QA[pt_key], alpha=0.85, edgecolor="white")

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(x + width)
ax.set_xticklabels(models, rotation=15, ha="right")
ax.set_ylabel("Exact Match delta (r=1 - r=0)")
ax.set_title("HotpotQA — EM Degradation Under Poisoning")
ax.legend(title="Prompt")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "hotpotqa_degradation.pdf", bbox_inches="tight")
plt.show()